In [4]:
import os
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
if Path.cwd().name == "notebooks":
    os.chdir("..")
print(Path.cwd())

/Users/majdalsuleh/Projects/mining-quality-prediction


In [10]:
con = duckdb.connect("data/mining.duckdb", read_only=True)
hourly = con.sql("""
    SELECT * FROM hourly
    WHERE hour >= TIMESTAMP '2017-03-29 12:00:00'
    ORDER BY hour
""").df()
con.close()

hourly = hourly.set_index("hour")
hourly.shape

(3948, 25)

In [11]:
hourly.index.to_series().diff().value_counts()

hour
0 days 01:00:00    3947
Name: count, dtype: int64

In [12]:
hourly["silica_known"] = (
    hourly["silica_conc"]
    .where(~hourly["is_filled"])   # hide filled-in values
    .ffill()                       # carry the last real value forward
)

In [13]:
hourly["target"] = hourly["silica_conc"].shift(-1)
hourly["target_is_real"] = ~hourly["is_filled"].shift(-1, fill_value=True)

print(hourly["target_is_real"].sum(), "hours with a real next-hour target")

3508 hours with a real next-hour target


In [14]:
s = hourly["silica_known"]
feat = pd.DataFrame(index=hourly.index)

feat["silica_now"]      = s                    # last known lab result
feat["silica_lag1"]     = s.shift(1)           # 1 hour earlier
feat["silica_lag2"]     = s.shift(2)           # 2 hours earlier
feat["silica_lag3"]     = s.shift(3)           # 3 hours earlier
feat["silica_mean_6h"]  = s.rolling(6).mean()  # average of last 6 hours
feat["silica_mean_24h"] = s.rolling(24).mean() # average of last 24 hours
feat["silica_std_6h"]   = s.rolling(6).std()   # how unstable the last 6 hours were

# How long since the lab last reported a real value (0 = this hour)
real = ~hourly["is_filled"]
feat["hours_since_lab"] = hourly.groupby(real.cumsum()).cumcount()

In [15]:
sensors = (
    ["iron_feed", "silica_feed", "starch_flow", "amina_flow",
     "pulp_flow", "pulp_ph", "pulp_density"]
    + [f"air_{i:02d}" for i in range(1, 8)]
    + [f"level_{i:02d}" for i in range(1, 8)]
)

for col in sensors:
    feat[col] = hourly[col]                     # this hour's average
    feat[f"{col}_chg"] = hourly[col].diff()     # change from last hour

In [16]:
feat["target"] = hourly["target"]

model_table = feat[hourly["target_is_real"]].dropna()
print(model_table.shape)

(3485, 51)


In [17]:
model_table.corr()["target"].drop("target").sort_values(key=abs, ascending=False).head(10).round(2)

silica_now         0.74
silica_lag1        0.62
silica_mean_6h     0.60
silica_lag2        0.53
silica_lag3        0.44
silica_mean_24h    0.40
silica_std_6h      0.26
level_04          -0.20
level_05          -0.20
amina_flow         0.19
Name: target, dtype: float64

In [18]:
model_table.to_csv("data/model_table.csv")